# 00 — Prepared PBMC inputs

**Objective.** Validate the standardized post-SCENIC+/cisTopic inputs used by CRANE-GRN. This tutorial intentionally begins after raw 10x processing and upstream model construction. For the upstream processing, please follow SCENIC+ guidelines.

**Inputs.** `scplusmdata.h5mu`, `cistopic_obj.pkl`, `AUCell_direct.h5mu`, and `eRegulon_direct.tsv`. The large upstream objects are inspected here but are never copied into tutorial outputs.

## Configuration and imports

The archive path is portable. Frozen manuscript outputs are read-only; newly generated files belong under `WORK_ROOT`.

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
import gc
import os
import pickle
from pathlib import Path

import mudata
import pandas as pd

from _tutorial_utils import ensure_crane_grn_importable, require_path

ensure_crane_grn_importable()
from crane_grn.io import extract_cistopic_matrices, get_modality

DATA_ROOT = Path(os.environ.get("CRANE_GRN_DATA_ROOT", "../../CRANE_GRN_data")).expanduser().resolve()
WORK_ROOT = Path(os.environ.get("CRANE_GRN_TUTORIAL_WORK", "./results")).expanduser().resolve()
PBMC_INPUT_ROOT = DATA_ROOT / "PBMC" / "prepared_inputs"
PBMC_REFERENCE_ROOT = DATA_ROOT / "PBMC" / "outputs"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

SCPLUS_PATH = require_path(PBMC_INPUT_ROOT / "scplusmdata.h5mu", "PBMC scplusmdata input")
CISTOPIC_PATH = require_path(PBMC_INPUT_ROOT / "cistopic_obj.pkl", "PBMC cisTopic input")
AUCELL_PATH = require_path(PBMC_REFERENCE_ROOT / "AUCell_direct.h5mu", "direct AUCell output")
EREGULON_PATH = require_path(PBMC_REFERENCE_ROOT / "eRegulon_direct.tsv", "direct eRegulon metadata")

## Inspect the SCENIC+ MuData object

CRANE-GRN uses raw RNA counts, observed ATAC accessibility, and SCENIC+ annotations. 

In [ ]:
scplus = mudata.read(str(SCPLUS_PATH))
rna = get_modality(scplus, "scRNA_counts")
atac = get_modality(scplus, "scATAC_counts")

expected_scplus = {"cells": 10908, "rna_genes": 26349, "atac_regions": 86330} 
observed_scplus = {
    "cells": int(rna.n_obs),
    "rna_genes": int(rna.n_vars),
    "atac_regions": int(atac.n_vars),
    "same_cell_set": set(rna.obs_names.astype(str)) == set(atac.obs_names.astype(str)),
}
print("Modalities:", list(scplus.mod.keys()))
print("Observed:", observed_scplus)
for key, expected in expected_scplus.items():
    if observed_scplus[key] != expected:
        raise ValueError("PBMC {} is {}; expected {}.".format(key, observed_scplus[key], expected))

## Inspect the selected cisTopic model

`extract_cistopic_matrices()` exposes the selected model as region-by-topic `L` and cell-by-topic `T`. The source helper preserves source ordering and validates the probability-like topic sums.

In [ ]:
with CISTOPIC_PATH.open("rb") as handle:
    cistopic = pickle.load(handle)

region_by_topic, cell_by_topic = extract_cistopic_matrices(cistopic)
topic_summary = {
    "selected_topics": int(cistopic.selected_model.n_topic),
    "region_by_topic": tuple(region_by_topic.shape),
    "cell_by_topic": tuple(cell_by_topic.shape),
    "region_axis_matches": len(region_by_topic.index) == len(cistopic.region_names),
    "cell_axis_matches": len(cell_by_topic.index) == len(cistopic.cell_names),
}
print(topic_summary)
if topic_summary["selected_topics"] != 50:
    raise ValueError("PBMC selected cisTopic model must contain 50 topics.")
if region_by_topic.shape != (86330, 50) or cell_by_topic.shape != (10908, 50):
    raise ValueError("Unexpected PBMC cisTopic matrix dimensions.")

## Inspect direct eRegulon inputs

Region-based AUCell is used during support aggregation. Gene-based AUCell is retained as a complementary SCENIC+ output. Both axes must describe the same 113 direct eRegulons.

In [ ]:
aucell_container = mudata.read(str(AUCELL_PATH))
gene_auc = get_modality(aucell_container, "Gene_based")
region_auc = get_modality(aucell_container, "Region_based")
eregulon_metadata = pd.read_csv(EREGULON_PATH, sep="\t")
required_metadata = {
    "Region", "Gene", "TF", "Region_signature_name",
    "Gene_signature_name", "regulation", "rho_TF2G",
    "rho_R2G", "triplet_rank",
}
missing_metadata = sorted(required_metadata.difference(eregulon_metadata.columns))
if missing_metadata:
    raise ValueError("Direct eRegulon metadata is missing: {}".format(missing_metadata))
if gene_auc.shape != (10908, 113) or region_auc.shape != (10908, 113):
    raise ValueError("Expected both direct AUCell modalities to have shape (10908, 113).")

print("Gene-based AUCell:", gene_auc.shape)
print("Region-based AUCell:", region_auc.shape)
print("Direct eRegulon metadata:", eregulon_metadata.shape)
display(eregulon_metadata.loc[:, sorted(required_metadata)].head(3))

## Checkpoints and interpretation

The validated boundary contains 10,908 paired cells, 26,349 RNA genes, 86,330 ATAC regions, 50 topics, and 113 direct eRegulons. SCENIC+ and cisTopic remain prerequisites, while CRANE-GRN begins from these standardized objects.

**Next step:** construct or load factor-region occupancy matrices and create cell-type-specific iRF inputs.